# 13 - Audit: Cross-Source Coverage

This notebook creates the cross-source coverage checkpoint before gold tables.

Outputs:

- `audit.country_year_source_coverage`
- `audit.source_coverage_summary`

The country-year coverage table has one row per project country per year from 1990 to 2024. It records whether macro, trade, ACLED conflict, and FSI fragility data are available for that country-year. This makes missing data explicit before dashboard marts are built.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS audit")

START_YEAR = 1990
END_YEAR = 2024
loaded_at = datetime.now(timezone.utc)

print("Target tables:")
print("  audit.country_year_source_coverage")
print("  audit.source_coverage_summary")
print(f"Years: {START_YEAR}-{END_YEAR}")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("silver", "dim_country"),
    ("silver", "dim_bloc_membership"),
    ("silver", "fact_macro_annual"),
    ("silver", "fact_trade_partner_annual"),
    ("silver", "trade_partner_concentration_annual"),
    ("silver", "fact_acled_country_year"),
    ("silver", "fact_fsi_annual"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")

print("Silver table row counts:")
for schema_name, table_name in required_tables:
    row_count = spark.table(f"{schema_name}.{table_name}").count()
    print(f"  {schema_name}.{table_name}: {row_count:,}")

In [ ]:
# Cell 3 - Build canonical country-year panel with time-aware analytical bloc
country_dim = spark.table("silver.dim_country").select(
    "country_key",
    "country_iso3",
    "country_name",
    F.col("region").alias("project_region"),
)

years_df = spark.range(START_YEAR, END_YEAR + 1).select(F.col("id").cast("int").alias("year"))

panel_df = (
    country_dim.crossJoin(years_df)
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
)

bloc_df = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("bloc_country_iso3"),
    F.col("bloc_code").alias("analytical_bloc_code"),
    F.col("bloc_name").alias("analytical_bloc_name"),
    F.col("valid_from").alias("bloc_valid_from"),
    F.col("valid_to").alias("bloc_valid_to"),
)

panel_with_bloc_df = (
    panel_df.alias("p")
    .join(
        bloc_df.alias("b"),
        (F.col("p.country_iso3") == F.col("b.bloc_country_iso3"))
        & (F.col("p.year_end_date") >= F.col("b.bloc_valid_from"))
        & (F.col("b.bloc_valid_to").isNull() | (F.col("p.year_end_date") <= F.col("b.bloc_valid_to"))),
        "left",
    )
    .select(
        F.col("p.country_key"),
        F.col("p.country_iso3"),
        F.col("p.country_name"),
        F.col("p.project_region"),
        F.col("p.year"),
        F.col("p.year_end_date"),
        F.col("b.analytical_bloc_code"),
        F.col("b.analytical_bloc_name"),
    )
)

expected_panel_rows = 21 * (END_YEAR - START_YEAR + 1)
actual_panel_rows = panel_with_bloc_df.count()
print(f"Country-year panel rows: {actual_panel_rows:,}")
if actual_panel_rows != expected_panel_rows:
    raise ValueError(f"Expected {expected_panel_rows} country-year rows, got {actual_panel_rows}")

missing_bloc_rows = panel_with_bloc_df.where(F.col("analytical_bloc_code").isNull()).count()
if missing_bloc_rows:
    raise ValueError(f"Country-year rows missing analytical bloc: {missing_bloc_rows}")

panel_with_bloc_df.groupBy("analytical_bloc_code").count().orderBy("analytical_bloc_code").show()

In [ ]:
# Cell 4 - Build source-specific coverage extracts
macro_coverage_df = spark.table("silver.fact_macro_annual").select(
    "country_iso3",
    "year",
    F.lit(True).alias("has_macro_row"),
    F.col("has_world_bank_macro"),
    F.col("has_imf_fiscal_context"),
    F.col("gdp_current_usd").isNotNull().alias("has_gdp"),
    F.col("population").isNotNull().alias("has_population"),
    F.col("inflation_cpi_pct").isNotNull().alias("has_inflation"),
    F.col("gross_debt_pct_gdp_imf").isNotNull().alias("has_debt"),
    F.col("missing_core_metric_count"),
    F.col("gdp_current_usd_billions"),
    F.col("population_millions"),
)

trade_coverage_df = (
    spark.table("silver.fact_trade_partner_annual")
    .groupBy(F.col("reporter_iso3").alias("country_iso3"), "year")
    .agg(
        F.count("*").alias("trade_rows"),
        F.sum(F.when(F.col("is_world_aggregate"), F.lit(1)).otherwise(F.lit(0))).alias("world_aggregate_trade_rows"),
        F.countDistinct(F.when(~F.col("is_world_aggregate"), F.col("counterpart_iso3"))).alias("trade_partner_count"),
        F.sum(F.when(~F.col("is_world_aggregate"), F.coalesce(F.col("exports_fob_usd"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("exports_partner_sum_usd"),
        F.sum(F.when(~F.col("is_world_aggregate"), F.coalesce(F.col("imports_cif_usd"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("imports_partner_sum_usd"),
        F.sum(F.when(~F.col("is_world_aggregate"), F.coalesce(F.col("total_trade_usd"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("total_trade_partner_sum_usd"),
    )
    .withColumn("has_trade_row", F.col("trade_rows") > 0)
    .withColumn("has_non_world_trade_partner", F.col("trade_partner_count") > 0)
)

trade_hhi_df = spark.table("silver.trade_partner_concentration_annual").select(
    F.col("reporter_iso3").alias("hhi_country_iso3"),
    F.col("year").alias("hhi_year"),
    "export_partner_hhi",
    "import_partner_hhi",
    "total_trade_partner_hhi",
)

acled_coverage_df = spark.table("silver.fact_acled_country_year").select(
    "country_iso3",
    "year",
    F.lit(True).alias("has_acled_row"),
    "total_events",
    "violent_events",
    "fatalities",
    "events_per_million",
    "fatalities_per_million",
)

fsi_coverage_df = spark.table("silver.fact_fsi_annual").select(
    "country_iso3",
    "year",
    F.lit(True).alias("has_fsi_row"),
    "fsi_rank",
    "fsi_total_score",
    "fragility_band",
)

print("Coverage extracts built.")

In [ ]:
# Cell 5 - Join coverage extracts and derive coverage flags
coverage_df = (
    panel_with_bloc_df.alias("p")
    .join(macro_coverage_df.alias("m"), ["country_iso3", "year"], "left")
    .join(trade_coverage_df.alias("t"), ["country_iso3", "year"], "left")
    .join(
        trade_hhi_df.alias("h"),
        (F.col("p.country_iso3") == F.col("h.hhi_country_iso3")) & (F.col("p.year") == F.col("h.hhi_year")),
        "left",
    )
    .join(acled_coverage_df.alias("a"), ["country_iso3", "year"], "left")
    .join(fsi_coverage_df.alias("f"), ["country_iso3", "year"], "left")
    .select(
        F.col("p.country_key"),
        F.col("p.country_iso3"),
        F.col("p.country_name"),
        F.col("p.project_region"),
        F.col("p.analytical_bloc_code"),
        F.col("p.analytical_bloc_name"),
        F.col("p.year"),
        F.col("p.year_end_date"),
        F.coalesce(F.col("m.has_macro_row"), F.lit(False)).alias("has_macro_row"),
        F.coalesce(F.col("m.has_world_bank_macro"), F.lit(False)).alias("has_world_bank_macro"),
        F.coalesce(F.col("m.has_imf_fiscal_context"), F.lit(False)).alias("has_imf_fiscal_context"),
        F.coalesce(F.col("m.has_gdp"), F.lit(False)).alias("has_gdp"),
        F.coalesce(F.col("m.has_population"), F.lit(False)).alias("has_population"),
        F.coalesce(F.col("m.has_inflation"), F.lit(False)).alias("has_inflation"),
        F.coalesce(F.col("m.has_debt"), F.lit(False)).alias("has_debt"),
        F.col("m.missing_core_metric_count"),
        F.col("m.gdp_current_usd_billions"),
        F.col("m.population_millions"),
        F.coalesce(F.col("t.has_trade_row"), F.lit(False)).alias("has_trade_row"),
        F.coalesce(F.col("t.has_non_world_trade_partner"), F.lit(False)).alias("has_non_world_trade_partner"),
        F.col("t.trade_rows"),
        F.col("t.world_aggregate_trade_rows"),
        F.col("t.trade_partner_count"),
        F.col("t.exports_partner_sum_usd"),
        F.col("t.imports_partner_sum_usd"),
        F.col("t.total_trade_partner_sum_usd"),
        F.col("h.export_partner_hhi"),
        F.col("h.import_partner_hhi"),
        F.col("h.total_trade_partner_hhi"),
        F.coalesce(F.col("a.has_acled_row"), F.lit(False)).alias("has_acled_row"),
        F.col("a.total_events"),
        F.col("a.violent_events"),
        F.col("a.fatalities"),
        F.col("a.events_per_million"),
        F.col("a.fatalities_per_million"),
        F.coalesce(F.col("f.has_fsi_row"), F.lit(False)).alias("has_fsi_row"),
        F.col("f.fsi_rank"),
        F.col("f.fsi_total_score"),
        F.col("f.fragility_band"),
    )
)

coverage_df = (
    coverage_df
    .withColumn("macro_source_present", F.col("has_macro_row") & F.col("has_gdp") & F.col("has_population"))
    .withColumn("trade_source_present", F.col("has_trade_row") & F.col("has_non_world_trade_partner"))
    .withColumn("acled_source_present", F.col("has_acled_row"))
    .withColumn("fsi_source_present", F.col("has_fsi_row"))
    .withColumn(
        "source_count",
        F.col("macro_source_present").cast("int")
        + F.col("trade_source_present").cast("int")
        + F.col("acled_source_present").cast("int")
        + F.col("fsi_source_present").cast("int"),
    )
    .withColumn(
        "coverage_level",
        F.when(F.col("source_count") == 4, F.lit("all_four_sources"))
        .when(F.col("source_count") == 3, F.lit("three_sources"))
        .when(F.col("source_count") == 2, F.lit("two_sources"))
        .when(F.col("source_count") == 1, F.lit("one_source"))
        .otherwise(F.lit("no_sources")),
    )
    .withColumn("dashboard_core_ready", F.col("macro_source_present") & F.col("trade_source_present") & F.col("acled_source_present"))
    .withColumn("full_fragility_context_ready", F.col("dashboard_core_ready") & F.col("fsi_source_present"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

print(f"Coverage rows: {coverage_df.count():,}")
coverage_df.groupBy("coverage_level").count().orderBy("coverage_level").show(truncate=False)

In [ ]:
# Cell 6 - Write audit.country_year_source_coverage
spark.sql("DROP TABLE IF EXISTS audit.country_year_source_coverage")

(coverage_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("audit.country_year_source_coverage"))

print("Write complete: audit.country_year_source_coverage")

In [ ]:
# Cell 7 - Build and write audit.source_coverage_summary
source_summary_df = spark.sql("""
    SELECT
        'macro' AS source_name,
        'silver.fact_macro_annual' AS source_table,
        SUM(CASE WHEN macro_source_present THEN 1 ELSE 0 END) AS covered_country_years,
        COUNT(*) AS possible_country_years,
        ROUND(100.0 * SUM(CASE WHEN macro_source_present THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_country_years_covered,
        COUNT(DISTINCT CASE WHEN macro_source_present THEN country_iso3 END) AS countries_covered,
        MIN(CASE WHEN macro_source_present THEN year END) AS earliest_year,
        MAX(CASE WHEN macro_source_present THEN year END) AS latest_year
    FROM audit.country_year_source_coverage

    UNION ALL

    SELECT
        'trade' AS source_name,
        'silver.fact_trade_partner_annual' AS source_table,
        SUM(CASE WHEN trade_source_present THEN 1 ELSE 0 END) AS covered_country_years,
        COUNT(*) AS possible_country_years,
        ROUND(100.0 * SUM(CASE WHEN trade_source_present THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_country_years_covered,
        COUNT(DISTINCT CASE WHEN trade_source_present THEN country_iso3 END) AS countries_covered,
        MIN(CASE WHEN trade_source_present THEN year END) AS earliest_year,
        MAX(CASE WHEN trade_source_present THEN year END) AS latest_year
    FROM audit.country_year_source_coverage

    UNION ALL

    SELECT
        'acled' AS source_name,
        'silver.fact_acled_country_year' AS source_table,
        SUM(CASE WHEN acled_source_present THEN 1 ELSE 0 END) AS covered_country_years,
        COUNT(*) AS possible_country_years,
        ROUND(100.0 * SUM(CASE WHEN acled_source_present THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_country_years_covered,
        COUNT(DISTINCT CASE WHEN acled_source_present THEN country_iso3 END) AS countries_covered,
        MIN(CASE WHEN acled_source_present THEN year END) AS earliest_year,
        MAX(CASE WHEN acled_source_present THEN year END) AS latest_year
    FROM audit.country_year_source_coverage

    UNION ALL

    SELECT
        'fsi' AS source_name,
        'silver.fact_fsi_annual' AS source_table,
        SUM(CASE WHEN fsi_source_present THEN 1 ELSE 0 END) AS covered_country_years,
        COUNT(*) AS possible_country_years,
        ROUND(100.0 * SUM(CASE WHEN fsi_source_present THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_country_years_covered,
        COUNT(DISTINCT CASE WHEN fsi_source_present THEN country_iso3 END) AS countries_covered,
        MIN(CASE WHEN fsi_source_present THEN year END) AS earliest_year,
        MAX(CASE WHEN fsi_source_present THEN year END) AS latest_year
    FROM audit.country_year_source_coverage
""").withColumn("loaded_at", F.lit(loaded_at))

spark.sql("DROP TABLE IF EXISTS audit.source_coverage_summary")

(source_summary_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("audit.source_coverage_summary"))

print("Write complete: audit.source_coverage_summary")
source_summary_df.orderBy("source_name").show(truncate=False)

In [ ]:
# Cell 8 - Verification and dashboard-readiness checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_country_years,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        SUM(CASE WHEN macro_source_present THEN 1 ELSE 0 END) AS macro_country_years,
        SUM(CASE WHEN trade_source_present THEN 1 ELSE 0 END) AS trade_country_years,
        SUM(CASE WHEN acled_source_present THEN 1 ELSE 0 END) AS acled_country_years,
        SUM(CASE WHEN fsi_source_present THEN 1 ELSE 0 END) AS fsi_country_years,
        SUM(CASE WHEN dashboard_core_ready THEN 1 ELSE 0 END) AS dashboard_core_ready_country_years,
        SUM(CASE WHEN full_fragility_context_ready THEN 1 ELSE 0 END) AS full_context_ready_country_years
    FROM audit.country_year_source_coverage
""").show()

print("Coverage by year:")
spark.sql("""
    SELECT
        year,
        SUM(CASE WHEN macro_source_present THEN 1 ELSE 0 END) AS macro_countries,
        SUM(CASE WHEN trade_source_present THEN 1 ELSE 0 END) AS trade_countries,
        SUM(CASE WHEN acled_source_present THEN 1 ELSE 0 END) AS acled_countries,
        SUM(CASE WHEN fsi_source_present THEN 1 ELSE 0 END) AS fsi_countries,
        SUM(CASE WHEN dashboard_core_ready THEN 1 ELSE 0 END) AS dashboard_core_ready_countries,
        SUM(CASE WHEN full_fragility_context_ready THEN 1 ELSE 0 END) AS full_context_ready_countries
    FROM audit.country_year_source_coverage
    GROUP BY year
    ORDER BY year
""").show(40, truncate=False)

print("Country-level gap summary:")
spark.sql("""
    SELECT
        country_iso3,
        country_name,
        analytical_bloc_code,
        SUM(CASE WHEN macro_source_present THEN 1 ELSE 0 END) AS macro_years,
        SUM(CASE WHEN trade_source_present THEN 1 ELSE 0 END) AS trade_years,
        SUM(CASE WHEN acled_source_present THEN 1 ELSE 0 END) AS acled_years,
        SUM(CASE WHEN fsi_source_present THEN 1 ELSE 0 END) AS fsi_years,
        SUM(CASE WHEN dashboard_core_ready THEN 1 ELSE 0 END) AS dashboard_core_ready_years,
        SUM(CASE WHEN full_fragility_context_ready THEN 1 ELSE 0 END) AS full_context_ready_years
    FROM audit.country_year_source_coverage
    GROUP BY country_iso3, country_name, analytical_bloc_code
    ORDER BY full_context_ready_years ASC, country_iso3
""").show(25, truncate=False)

print("Latest-year dashboard coverage:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        country_iso3,
        country_name,
        year,
        macro_source_present,
        trade_source_present,
        acled_source_present,
        fsi_source_present,
        dashboard_core_ready,
        full_fragility_context_ready,
        ROUND(gdp_current_usd_billions, 2) AS gdp_billions_usd,
        trade_partner_count,
        total_events,
        ROUND(fsi_total_score, 1) AS fsi_total_score
    FROM audit.country_year_source_coverage
    WHERE year = (SELECT MAX(year) FROM audit.country_year_source_coverage)
    ORDER BY analytical_bloc_code, country_iso3
""").show(25, truncate=False)